<a href="https://colab.research.google.com/github/noel-odero/Zindi-Health_QA/blob/main/zindi_colab_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - Mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q \
    'accelerate>=1.1.0' \
    'bitsandbytes>=0.46.1' \
    peft==0.12.0 \
    datasets==2.21.0 \
    rouge-score \
    sentencepiece \
    tqdm

print('Done - restart runtime now if this is the first time running')

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
Done - restart runtime now if this is the first time running


In [2]:
# Cell 2 - Imports and config
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF']     = 'expandable_segments:True'

import re
import gc
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    set_seed,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    prepare_model_for_kbit_training,
    PeftModel,
)
from datasets import Dataset as HFDataset
from rouge_score import rouge_scorer

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# Paths - all outputs saved to Drive so disconnections don't lose progress
DRIVE_DIR  = Path('/content/drive/MyDrive/zindi-health-qa')
DATA_DIR   = DRIVE_DIR
OUT_DIR    = DRIVE_DIR / 'outputs'
CKPT_DIR   = DRIVE_DIR / 'checkpoints'
PRED_DIR   = DRIVE_DIR / 'predictions'

for d in [OUT_DIR, CKPT_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PATH   = DATA_DIR / 'Train.csv'
VAL_PATH     = DATA_DIR / 'Val.csv'
TEST_PATH    = DATA_DIR / 'Test.csv'
SUMMARY_PATH = OUT_DIR / 'experiment_summary.csv'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device   : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU - go to Runtime > Change runtime type > T4 GPU')
print('Done')

Device   : cuda
GPU      : Tesla T4
VRAM     : 15.6 GB
Done


In [3]:
# Cell 3 - Load and clean data
LANG_MAP = {
    'Aka': 'Akan',
    'Amh': 'Amharic',
    'Eng': 'English',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
}

MAX_NEW_TOKENS_MAP = {
    'Aka_Gha': 320,
    'Amh_Eth': 60,
    'Eng_Eth': 70,
    'Eng_Gha': 230,
    'Eng_Ken': 260,
    'Eng_Uga': 350,
    'Lug_Uga': 270,
    'Swa_Ken': 300,
}
DEFAULT_MAX_NEW_TOKENS = 300

def get_language_name(subset):
    return LANG_MAP.get(subset.split('_')[0], 'English')

def clean_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()

def build_prompt(question, subset, use_lang_tag=True):
    if use_lang_tag:
        lang = get_language_name(subset)
        return f'Answer the following health question in {lang}: {question.strip()}'
    return f'Answer the following health question: {question.strip()}'

train_raw = pd.read_csv(TRAIN_PATH)
val_raw   = pd.read_csv(VAL_PATH)
test_raw  = pd.read_csv(TEST_PATH)

for df in [train_raw, val_raw, test_raw]:
    df['input'] = df['input'].map(clean_text)
    if 'output' in df.columns:
        df['output'] = df['output'].map(clean_text)

train_raw = train_raw[
    (train_raw['input'] != '') & (train_raw['output'] != '')
].reset_index(drop=True)
val_raw   = val_raw[
    (val_raw['input'] != '') & (val_raw['output'] != '')
].reset_index(drop=True)
test_raw  = test_raw[test_raw['input'] != ''].reset_index(drop=True)

# Deduplicated version - keep longer answer
train_raw['answer_len'] = train_raw['output'].str.len()
train_dedup = (
    train_raw
    .sort_values('answer_len', ascending=False)
    .drop_duplicates(subset=['input', 'subset'], keep='first')
    .drop(columns='answer_len')
    .reset_index(drop=True)
)
train_raw = train_raw.drop(columns='answer_len').reset_index(drop=True)

print(f'Train dedup : {len(train_dedup)} rows')
print(f'Train raw   : {len(train_raw)} rows')
print(f'Val         : {len(val_raw)} rows')
print(f'Test        : {len(test_raw)} rows')
print('Done')

Train dedup : 28834 rows
Train raw   : 29814 rows
Val         : 6686 rows
Test        : 2618 rows
Done


In [4]:
# Cell 4 - ROUGE utilities and experiment tracker
class WhitespaceTokenizer:
    def tokenize(self, text):
        return str(text).strip().split() if text else []

_scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rougeL'],
    tokenizer=WhitespaceTokenizer(),
    use_stemmer=False,
)

def compute_rouge(predictions, references):
    r1, rl = [], []
    for pred, ref in zip(predictions, references):
        s = _scorer.score(str(ref), str(pred))
        r1.append(s['rouge1'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {
        'rouge1': float(np.mean(r1)),
        'rougeL': float(np.mean(rl)),
    }

def evaluate_by_language(predictions, references, subsets):
    rows = []
    for lang in sorted(set(subsets)):
        mask  = [s == lang for s in subsets]
        preds = [p for p, m in zip(predictions, mask) if m]
        refs  = [r for r, m in zip(references,  mask) if m]
        m = compute_rouge(preds, refs)
        rows.append({
            'subset' : lang,
            'n'      : len(preds),
            'ROUGE-1': round(m['rouge1'], 4),
            'ROUGE-L': round(m['rougeL'], 4),
        })
    overall = compute_rouge(predictions, references)
    rows.append({
        'subset' : 'OVERALL',
        'n'      : len(predictions),
        'ROUGE-1': round(overall['rouge1'], 4),
        'ROUGE-L': round(overall['rougeL'], 4),
    })
    return pd.DataFrame(rows).set_index('subset')

# Load existing summary if resuming after disconnect
if SUMMARY_PATH.exists():
    experiment_log = pd.read_csv(SUMMARY_PATH).to_dict('records')
    print(f'Loaded {len(experiment_log)} existing experiment records')
else:
    experiment_log = []

def log_experiment(exp_id, name, description, rouge1, rougeL, notes=''):
    weighted = round(0.37 * rouge1 + 0.37 * rougeL, 4)
    # Remove existing entry for this exp_id if resuming
    global experiment_log
    experiment_log = [e for e in experiment_log if e['exp_id'] != exp_id]
    entry = {
        'exp_id'     : exp_id,
        'name'       : name,
        'description': description,
        'rouge1'     : round(rouge1, 4),
        'rougeL'     : round(rougeL, 4),
        'weighted'   : weighted,
        'notes'      : notes,
        'timestamp'  : datetime.now().strftime('%Y-%m-%d %H:%M'),
    }
    experiment_log.append(entry)
    pd.DataFrame(experiment_log).to_csv(SUMMARY_PATH, index=False)
    print(f'Exp {exp_id} | {name}')
    print(f'  ROUGE-1  : {rouge1:.4f}')
    print(f'  ROUGE-L  : {rougeL:.4f}')
    print(f'  Weighted : {weighted:.4f}')
    return entry

print('ROUGE utilities and experiment tracker ready')

Loaded 1 existing experiment records
ROUGE utilities and experiment tracker ready


In [5]:
# Cell 5 - All reusable utilities
MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 384

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_base_model(model_name, use_4bit=True):
    print(f'Loading {model_name} ...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_quant_type       = 'nf4',
            bnb_4bit_compute_dtype    = torch.bfloat16,
            bnb_4bit_use_double_quant = True,
        )
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            quantization_config = bnb_config,
        )
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype = torch.bfloat16,
        ).to('cuda')

    used = torch.cuda.memory_allocated(0) / 1e9
    free = torch.cuda.get_device_properties(0).total_memory / 1e9 - used
    print(f'  Params : {sum(p.numel() for p in model.parameters())/1e9:.2f}B')
    print(f'  VRAM   : used={used:.1f}GB  free={free:.1f}GB')
    return model, tokenizer

def apply_lora(model, lora_r=16):
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )
    lora_config = LoraConfig(
        task_type      = TaskType.SEQ_2_SEQ_LM,
        r              = lora_r,
        lora_alpha     = lora_r * 2,
        lora_dropout   = 0.05,
        bias           = 'none',
        target_modules = ['q', 'v'],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

def make_hf_dataset(df, tokenizer, use_lang_tag=True, is_train=True):
    records = []
    for _, row in df.iterrows():
        prompt = build_prompt(row['input'], row['subset'], use_lang_tag)
        record = {'prompt': prompt}
        if is_train:
            record['answer'] = str(row['output'])
        records.append(record)

    raw_ds = HFDataset.from_list(records)

    def preprocess(examples):
        model_inputs = tokenizer(
            examples['prompt'],
            max_length = MAX_INPUT_LEN,
            truncation = True,
            padding    = False,
        )
        if 'answer' in examples:
            labels = tokenizer(
                text_target = examples['answer'],
                max_length  = MAX_TARGET_LEN,
                truncation  = True,
                padding     = False,
            )
            model_inputs['labels'] = [
                [(t if t != tokenizer.pad_token_id else -100) for t in seq]
                for seq in labels['input_ids']
            ]
        return model_inputs

    remove_cols = [c for c in ['prompt', 'answer'] if c in raw_ds.column_names]
    return raw_ds.map(preprocess, batched=True, remove_columns=remove_cols)

def train_model(
    model, tokenizer, train_df, val_df, exp_name,
    epochs=3, batch_size=2, grad_accum=32, lr=3e-4, use_lang_tag=True,
):
    ckpt_path = CKPT_DIR / exp_name
    ckpt_path.mkdir(parents=True, exist_ok=True)

    hf_train = make_hf_dataset(train_df, tokenizer, use_lang_tag=use_lang_tag, is_train=True)
    hf_val   = make_hf_dataset(val_df,   tokenizer, use_lang_tag=use_lang_tag, is_train=True)

    data_collator = DataCollatorForSeq2Seq(
        tokenizer          = tokenizer,
        model              = model,
        label_pad_token_id = -100,
        pad_to_multiple_of = 8,
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir                  = str(ckpt_path),
        num_train_epochs            = epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        gradient_accumulation_steps = grad_accum,
        learning_rate               = lr,
        warmup_steps                = 100,
        lr_scheduler_type           = 'cosine',
        bf16                        = True,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        greater_is_better           = False,
        logging_steps               = 50,
        predict_with_generate       = False,
        report_to                   = 'none',
        seed                        = SEED,
    )

    trainer = Seq2SeqTrainer(
        model            = model,
        args             = training_args,
        train_dataset    = hf_train,
        eval_dataset     = hf_val,
        processing_class = tokenizer,
        data_collator    = data_collator,
    )

    print(f'Training {exp_name} on {len(hf_train):,} examples for {epochs} epochs')
    print(f'Effective batch size : {batch_size * grad_accum}')
    trainer.train()

    best_path = ckpt_path / 'best'
    model.save_pretrained(str(best_path))
    tokenizer.save_pretrained(str(best_path))
    print(f'Checkpoint saved to {best_path}')
    return model

def generate_answers(model, tokenizer, df, use_lang_tag=True, batch_size=4, num_beams=4):
    model.eval()
    all_answers = []
    rows = df.reset_index(drop=True)

    for start in tqdm(range(0, len(rows), batch_size), desc='Generating'):
        batch   = rows.iloc[start : start + batch_size]
        prompts = [
            build_prompt(r['input'], r['subset'], use_lang_tag)
            for _, r in batch.iterrows()
        ]
        subsets = batch['subset'].tolist()
        max_tok = max(
            MAX_NEW_TOKENS_MAP.get(s, DEFAULT_MAX_NEW_TOKENS) for s in subsets
        )

        inputs = tokenizer(
            prompts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = MAX_INPUT_LEN,
        ).to('cuda')

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens       = max_tok,
                num_beams            = num_beams,
                no_repeat_ngram_size = 3,
                early_stopping       = True,
            )

        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        cleaned = [re.sub(r'<extra_id_\d+>', '', a).strip() for a in decoded]
        all_answers.extend(cleaned)
        torch.cuda.empty_cache()

    return all_answers

def save_predictions(df, preds, exp_name):
    out = df[['ID', 'input', 'subset']].copy()
    out['prediction'] = preds
    if 'output' in df.columns:
        out['reference'] = df['output'].values
    path = PRED_DIR / f'{exp_name}_val_preds.csv'
    out.to_csv(path, index=False)
    print(f'Predictions saved to {path}')

def save_submission(model, tokenizer, exp_name, use_lang_tag=True, num_beams=4):
    print(f'Generating test predictions for {exp_name} ...')
    test_preds = generate_answers(model, tokenizer, test_raw, use_lang_tag=use_lang_tag, batch_size=4, num_beams=num_beams)

    clean_preds = [re.sub(r'<extra_id_\\d+>', '', str(p)).strip() for p in test_preds]
    clean_preds = [p if p else 'Please consult a qualified health professional.' for p in clean_preds]

    sub = pd.DataFrame({
        'ID'         : test_raw['ID'].values,
        'TargetRLF1' : clean_preds,
        'TargetR1F1' : clean_preds,
        'TargetLLM'  : clean_preds,
    })

    assert len(sub) == len(test_raw)
    assert sub[['TargetRLF1','TargetR1F1','TargetLLM']].notna().all().all()
    assert (sub['TargetRLF1'] == sub['TargetR1F1']).all()
    assert (sub['TargetRLF1'] == sub['TargetLLM']).all()
    assert (sub['TargetRLF1'].str.len() > 0).all()

    sub_path = OUT_DIR / f'{exp_name}_submission.csv'
    sub.to_csv(sub_path, index=False, encoding='utf-8')
    print(f'Submission saved to {sub_path}')
    return sub_path

print('All utilities ready')

All utilities ready


In [ ]:
# Cell 6 - Experiment 1: mt5-base zero-shot (no training)
EXP_ID    = 1
EXP_NAME  = 'exp1_mt5base_zeroshot'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    preds = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')['prediction'].tolist()
else:
    free_memory()
    model, tokenizer = load_base_model('google/mt5-base', use_4bit=False)

    print('Running zero-shot inference on val set')
    preds = generate_answers(model, tokenizer, val_raw, use_lang_tag=True, batch_size=8, num_beams=4)
    save_predictions(val_raw, preds, EXP_NAME)
    save_submission(model, tokenizer, EXP_NAME, use_lang_tag=True, num_beams=4)
    DONE_FLAG.touch()

    del model
    free_memory()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    'mt5-base zero-shot, no fine-tuning, beam=4, with lang tag',
    scores['rouge1'], scores['rougeL'],
    notes='Baseline - model performance without any task adaptation'
)
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())

In [ ]:
# Cell 7 - Experiment 2: mt5-base full fine-tune (no LoRA)
EXP_ID    = 2
EXP_NAME  = 'exp2_mt5base_fullft'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    preds = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')['prediction'].tolist()
else:
    free_memory()
    model, tokenizer = load_base_model('google/mt5-base', use_4bit=True)
    model.gradient_checkpointing_enable()

    model = train_model(
        model, tokenizer,
        train_df     = train_dedup,
        val_df       = val_raw,
        exp_name     = EXP_NAME,
        epochs       = 3,
        batch_size   = 2,
        grad_accum   = 32,
        lr           = 3e-4,
        use_lang_tag = True,
    )

    preds = generate_answers(model, tokenizer, val_raw, use_lang_tag=True, batch_size=4, num_beams=4)
    save_predictions(val_raw, preds, EXP_NAME)
    save_submission(model, tokenizer, EXP_NAME, use_lang_tag=True, num_beams=4)
    DONE_FLAG.touch()

    del model
    free_memory()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    'mt5-base full fine-tune, 4-bit, no LoRA, 3 epochs, beam=4, dedup train',
    scores['rouge1'], scores['rougeL'],
    notes='Full fine-tune - all params trained, higher memory cost than LoRA'
)
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())

In [ ]:
# Cell 8 - Experiment 3: mt5-base + LoRA r=8
EXP_ID    = 3
EXP_NAME  = 'exp3_mt5base_lora_r8'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    preds = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')['prediction'].tolist()
else:
    free_memory()
    model, tokenizer = load_base_model('google/mt5-base', use_4bit=False)
    model = apply_lora(model, lora_r=8)

    model = train_model(
        model, tokenizer,
        train_df     = train_dedup,
        val_df       = val_raw,
        exp_name     = EXP_NAME,
        epochs       = 2,
        batch_size   = 2,
        grad_accum   = 32,
        lr           = 3e-4,
        use_lang_tag = True,
    )

    preds = generate_answers(model, tokenizer, val_raw, use_lang_tag=True, batch_size=4, num_beams=4)
    save_predictions(val_raw, preds, EXP_NAME)
    save_submission(model, tokenizer, EXP_NAME, use_lang_tag=True, num_beams=4)
    DONE_FLAG.touch()

    del model
    free_memory()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    'mt5-base + LoRA r=8, 4-bit, 3 epochs, beam=4, dedup train',
    scores['rouge1'], scores['rougeL'],
    notes='Lower rank LoRA - fewer trainable params, faster but less capacity'
)
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())

Loading google/mt5-base ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


  Params : 0.58B
  VRAM   : used=3.2GB  free=12.5GB
trainable params: 884,736 || all params: 583,286,016 || trainable%: 0.1517


Map:   0%|          | 0/28834 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Training exp3_mt5base_lora_r8 on 28,834 examples for 2 epochs
Effective batch size : 64


Epoch,Training Loss,Validation Loss


In [ ]:
# Cell 9 - Experiment 4: mt5-base + LoRA r=16
EXP_ID    = 4
EXP_NAME  = 'exp4_mt5base_lora_r16'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    preds = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')['prediction'].tolist()
else:
    free_memory()
    model, tokenizer = load_base_model('google/mt5-base', use_4bit=True)
    model = apply_lora(model, lora_r=16)

    model = train_model(
        model, tokenizer,
        train_df     = train_dedup,
        val_df       = val_raw,
        exp_name     = EXP_NAME,
        epochs       = 3,
        batch_size   = 2,
        grad_accum   = 32,
        lr           = 3e-4,
        use_lang_tag = True,
    )

    preds = generate_answers(model, tokenizer, val_raw, use_lang_tag=True, batch_size=4, num_beams=4)
    save_predictions(val_raw, preds, EXP_NAME)
    save_submission(model, tokenizer, EXP_NAME, use_lang_tag=True, num_beams=4)
    DONE_FLAG.touch()

    del model
    free_memory()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    'mt5-base + LoRA r=16, 4-bit, 3 epochs, beam=4, dedup train',
    scores['rouge1'], scores['rougeL'],
    notes='Higher rank LoRA - more expressive adapters vs r=8'
)
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())

In [ ]:
# Cell 10 - Experiment 5: mt5-base + LoRA r=16, no language tag in prompt
EXP_ID    = 5
EXP_NAME  = 'exp5_mt5base_lora_r16_nolang'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    preds = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')['prediction'].tolist()
else:
    free_memory()
    model, tokenizer = load_base_model('google/mt5-base', use_4bit=True)
    model = apply_lora(model, lora_r=16)

    model = train_model(
        model, tokenizer,
        train_df     = train_dedup,
        val_df       = val_raw,
        exp_name     = EXP_NAME,
        epochs       = 3,
        batch_size   = 2,
        grad_accum   = 32,
        lr           = 3e-4,
        use_lang_tag = False,
    )

    preds = generate_answers(model, tokenizer, val_raw, use_lang_tag=False, batch_size=4, num_beams=4)
    save_predictions(val_raw, preds, EXP_NAME)
    save_submission(model, tokenizer, EXP_NAME, use_lang_tag=False, num_beams=4)
    DONE_FLAG.touch()

    del model
    free_memory()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    'mt5-base + LoRA r=16, 4-bit, 3 epochs, beam=4, no language tag',
    scores['rouge1'], scores['rougeL'],
    notes='Ablation: does naming the language in the prompt improve performance?'
)
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())

In [ ]:
# Cell 11 - Experiment 6: mt5-base + LoRA r=16, no deduplication
EXP_ID    = 6
EXP_NAME  = 'exp6_mt5base_lora_r16_nodedup'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    preds = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')['prediction'].tolist()
else:
    free_memory()
    model, tokenizer = load_base_model('google/mt5-base', use_4bit=True)
    model = apply_lora(model, lora_r=16)

    model = train_model(
        model, tokenizer,
        train_df     = train_raw,
        val_df       = val_raw,
        exp_name     = EXP_NAME,
        epochs       = 3,
        batch_size   = 2,
        grad_accum   = 32,
        lr           = 3e-4,
        use_lang_tag = True,
    )

    preds = generate_answers(model, tokenizer, val_raw, use_lang_tag=True, batch_size=4, num_beams=4)
    save_predictions(val_raw, preds, EXP_NAME)
    save_submission(model, tokenizer, EXP_NAME, use_lang_tag=True, num_beams=4)
    DONE_FLAG.touch()

    del model
    free_memory()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    'mt5-base + LoRA r=16, 4-bit, 3 epochs, beam=4, raw train no dedup',
    scores['rouge1'], scores['rougeL'],
    notes='Ablation: does deduplication improve quality?'
)
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())

In [ ]:
# Cell 12 - Experiment 7: beam search ablation on best model so far (exp4)
EXP_ID    = 7
EXP_NAME  = 'exp7_beam_ablation'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'
BEST_CKPT = CKPT_DIR / 'exp4_mt5base_lora_r16' / 'best'

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    saved = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')
    preds     = saved['prediction'].tolist()
    best_beam = int(saved['best_beam'].iloc[0])
else:
    free_memory()
    base_model, tokenizer = load_base_model('google/mt5-base', use_4bit=True)
    model = PeftModel.from_pretrained(base_model, str(BEST_CKPT))
    model.eval()

    beam_results = {}
    for num_beams in [2, 4, 6]:
        print(f'Testing beam={num_beams} ...')
        p = generate_answers(model, tokenizer, val_raw, use_lang_tag=True, batch_size=4, num_beams=num_beams)
        s = compute_rouge(p, val_raw['output'].tolist())
        beam_results[num_beams] = {'preds': p, 'rouge1': s['rouge1'], 'rougeL': s['rougeL']}
        print(f'  beam={num_beams}  ROUGE-1={s["rouge1"]:.4f}  ROUGE-L={s["rougeL"]:.4f}')

    best_beam = max(beam_results, key=lambda b: beam_results[b]['rouge1'])
    preds = beam_results[best_beam]['preds']

    out = val_raw[['ID', 'input', 'subset', 'output']].copy()
    out['prediction'] = preds
    out['best_beam']  = best_beam
    out.to_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv', index=False)
    save_submission(model, tokenizer, EXP_NAME, use_lang_tag=True, num_beams=best_beam)
    DONE_FLAG.touch()

    del model, base_model
    free_memory()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    f'Beam search ablation on exp4 model, best beam={best_beam}',
    scores['rouge1'], scores['rougeL'],
    notes=f'Tested beam=2,4,6 on fixed model - best was beam={best_beam}'
)
print(f'Best beam width: {best_beam}')
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())

In [ ]:
# Cell 13 - Experiment 8: epochs ablation (2 vs 3 vs 5)
EXP_ID    = 8
EXP_NAME  = 'exp8_epochs_ablation'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    saved       = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')
    preds       = saved['prediction'].tolist()
    best_epochs = int(saved['best_epochs'].iloc[0])
else:
    epoch_results = {}
    best_model_obj = None
    best_tokenizer_obj = None
    best_score_so_far = -1

    for n_epochs in [2, 3, 5]:
        print(f'Training with epochs={n_epochs} ...')
        free_memory()
        model, tokenizer = load_base_model('google/mt5-base', use_4bit=True)
        model = apply_lora(model, lora_r=16)

        model = train_model(
            model, tokenizer,
            train_df     = train_dedup,
            val_df       = val_raw,
            exp_name     = f'{EXP_NAME}_ep{n_epochs}',
            epochs       = n_epochs,
            batch_size   = 2,
            grad_accum   = 32,
            lr           = 3e-4,
            use_lang_tag = True,
        )

        p = generate_answers(model, tokenizer, val_raw, use_lang_tag=True, batch_size=4, num_beams=4)
        s = compute_rouge(p, val_raw['output'].tolist())
        epoch_results[n_epochs] = {'preds': p, 'rouge1': s['rouge1'], 'rougeL': s['rougeL']}
        print(f'  epochs={n_epochs}  ROUGE-1={s["rouge1"]:.4f}  ROUGE-L={s["rougeL"]:.4f}')

        if s['rouge1'] > best_score_so_far:
            best_score_so_far = s['rouge1']
            del best_model_obj
            free_memory()
            best_model_obj = model
            best_tokenizer_obj = tokenizer
        else:
            del model
            free_memory()

    best_epochs = max(epoch_results, key=lambda e: epoch_results[e]['rouge1'])
    preds = epoch_results[best_epochs]['preds']

    out = val_raw[['ID', 'input', 'subset', 'output']].copy()
    out['prediction']  = preds
    out['best_epochs'] = best_epochs
    out.to_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv', index=False)
    save_submission(best_model_obj, best_tokenizer_obj, EXP_NAME, use_lang_tag=True, num_beams=4)
    DONE_FLAG.touch()

    del best_model_obj, best_tokenizer_obj
    free_memory()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    f'Epochs ablation on mt5-base LoRA r=16, best epochs={best_epochs}',
    scores['rouge1'], scores['rougeL'],
    notes=f'Tested epochs=2,3,5 - best was {best_epochs}'
)
print(f'Best epochs: {best_epochs}')
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())


In [ ]:
# Cell 14 - Experiment 9: mt5-large + LoRA r=16 with best config
best_beam   = int(pd.read_csv(PRED_DIR / 'exp7_beam_ablation_val_preds.csv')['best_beam'].iloc[0])
best_epochs = int(pd.read_csv(PRED_DIR / 'exp8_epochs_ablation_val_preds.csv')['best_epochs'].iloc[0])
print(f'Using best beam={best_beam}, best epochs={best_epochs}')

EXP_ID    = 9
EXP_NAME  = 'exp9_mt5large_lora_r16'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    preds = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')['prediction'].tolist()
else:
    free_memory()
    model, tokenizer = load_base_model('google/mt5-large', use_4bit=True)
    model = apply_lora(model, lora_r=16)

    model = train_model(
        model, tokenizer,
        train_df     = train_dedup,
        val_df       = val_raw,
        exp_name     = EXP_NAME,
        epochs       = best_epochs,
        batch_size   = 2,
        grad_accum   = 32,
        lr           = 3e-4,
        use_lang_tag = True,
    )

    preds = generate_answers(model, tokenizer, val_raw, use_lang_tag=True, batch_size=4, num_beams=best_beam)
    save_predictions(val_raw, preds, EXP_NAME)
    save_submission(model, tokenizer, EXP_NAME, use_lang_tag=True, num_beams=4)
    DONE_FLAG.touch()

    del model
    free_memory()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    f'mt5-large + LoRA r=16, 4-bit, epochs={best_epochs}, beam={best_beam}, dedup train',
    scores['rouge1'], scores['rougeL'],
    notes='Bigger model with best config from previous experiments'
)
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())

In [ ]:
# Cell 15 - Experiment 10: best config retrained on train+val combined
summary     = pd.read_csv(SUMMARY_PATH)
best_beam   = int(pd.read_csv(PRED_DIR / 'exp7_beam_ablation_val_preds.csv')['best_beam'].iloc[0])
best_epochs = int(pd.read_csv(PRED_DIR / 'exp8_epochs_ablation_val_preds.csv')['best_epochs'].iloc[0])

exp9_score  = summary[summary['exp_id'] == 9]['rouge1'].values[0]
exp4_score  = summary[summary['exp_id'] == 4]['rouge1'].values[0]
best_model  = 'google/mt5-large' if exp9_score >= exp4_score else 'google/mt5-base'
print(f'Best model : {best_model}')
print(f'Best beam  : {best_beam}')
print(f'Best epochs: {best_epochs}')

EXP_ID    = 10
EXP_NAME  = 'exp10_final_trainval'
DONE_FLAG = CKPT_DIR / f'{EXP_NAME}.done'

full_train = pd.concat([train_dedup, val_raw], ignore_index=True)
print(f'Full train size: {len(full_train)} rows')

if DONE_FLAG.exists():
    print(f'Exp {EXP_ID} already done, loading saved predictions')
    preds = pd.read_csv(PRED_DIR / f'{EXP_NAME}_val_preds.csv')['prediction'].tolist()
else:
    free_memory()
    model, tokenizer = load_base_model(best_model, use_4bit=True)
    model = apply_lora(model, lora_r=16)

    model = train_model(
        model, tokenizer,
        train_df     = full_train,
        val_df       = val_raw,
        exp_name     = EXP_NAME,
        epochs       = best_epochs,
        batch_size   = 2,
        grad_accum   = 32,
        lr           = 3e-4,
        use_lang_tag = True,
    )

    preds = generate_answers(model, tokenizer, val_raw, use_lang_tag=True, batch_size=4, num_beams=best_beam)
    save_predictions(val_raw, preds, EXP_NAME)
    save_submission(model, tokenizer, EXP_NAME, use_lang_tag=True, num_beams=4)
    DONE_FLAG.touch()

scores = compute_rouge(preds, val_raw['output'].tolist())
log_experiment(
    EXP_ID, EXP_NAME,
    f'Final model on train+val, {best_model}, LoRA r=16, epochs={best_epochs}, beam={best_beam}',
    scores['rouge1'], scores['rougeL'],
    notes='Final submission model trained on all available labelled data'
)
print(evaluate_by_language(preds, val_raw['output'].tolist(), val_raw['subset'].tolist()).to_string())

In [ ]:
# Cell 16 - Generate test predictions and save submission
# Reload model from checkpoint if needed after disconnect
if 'model' not in vars() or model is None:
    print('Reloading final model from checkpoint ...')
    summary    = pd.read_csv(SUMMARY_PATH)
    exp9_score = summary[summary['exp_id'] == 9]['rouge1'].values[0]
    exp4_score = summary[summary['exp_id'] == 4]['rouge1'].values[0]
    best_model = 'google/mt5-large' if exp9_score >= exp4_score else 'google/mt5-base'

    free_memory()
    base_model, tokenizer = load_base_model(best_model, use_4bit=True)
    model = PeftModel.from_pretrained(base_model, str(CKPT_DIR / 'exp10_final_trainval' / 'best'))
    model.eval()
    print('Model reloaded')

best_beam = int(pd.read_csv(PRED_DIR / 'exp7_beam_ablation_val_preds.csv')['best_beam'].iloc[0])

print(f'Generating {len(test_raw)} test predictions ...')
test_preds = generate_answers(model, tokenizer, test_raw, use_lang_tag=True, batch_size=4, num_beams=best_beam)

clean_preds = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in test_preds]
clean_preds = [p if p else 'Please consult a qualified health professional.' for p in clean_preds]

sub = pd.DataFrame({
    'ID'         : test_raw['ID'].values,
    'TargetRLF1' : clean_preds,
    'TargetR1F1' : clean_preds,
    'TargetLLM'  : clean_preds,
})

assert len(sub) == len(test_raw)
assert sub[['TargetRLF1','TargetR1F1','TargetLLM']].notna().all().all()
assert (sub['TargetRLF1'] == sub['TargetR1F1']).all()
assert (sub['TargetRLF1'] == sub['TargetLLM']).all()
assert (sub['TargetRLF1'].str.len() > 0).all()

sub_path = OUT_DIR / 'submission.csv'
sub.to_csv(sub_path, index=False, encoding='utf-8')
print(f'Submission saved to {sub_path}')
print(sub.head(3).to_string())

In [ ]:
# Cell 17 - Print full experiment summary
summary = pd.read_csv(SUMMARY_PATH)
print('Experiment Summary')
print(summary[['exp_id','name','rouge1','rougeL','weighted','notes']].to_string(index=False))
best = summary.loc[summary['rouge1'].idxmax()]
print(f'\nBest experiment : {best["name"]}')
print(f'ROUGE-1         : {best["rouge1"]:.4f}')
print(f'ROUGE-L         : {best["rougeL"]:.4f}')
print(f'Weighted        : {best["weighted"]:.4f}')